# Ops Insight Assistant
### An Evidence-Based AI Agent for Data Pipeline Operations

---

## What This Notebook Is

This notebook walks through the full architecture and implementation of an AI-powered pipeline operations assistant — built from scratch in Python.

By the end you will understand:
- How to design a data pipeline that produces clean audit trails
- How to expose pipeline outputs to an AI agent through tool functions
- How to build an agent loop that reasons over evidence instead of guessing
- Why this pattern is used in production AI systems

---

## The Problem This Solves

Every company that runs data pipelines has the same operational challenge:

> "The pipeline ran overnight. Something looks off. Why did it fail? What changed from yesterday? What should we fix first?"

The traditional answer is: an engineer digs through logs, CSVs, and summary files manually. That takes time, requires context, and happens at 2am when something breaks.

The better answer: an AI agent that reads those same files through structured tools and produces a clear, evidence-backed answer in seconds.

---

## What Makes This Different From a Chatbot

A chatbot answers from general knowledge and can hallucinate.

This agent **cannot answer without first reading evidence**. Every claim in the response maps to a specific tool output. The model reasons over facts you can verify yourself.

```
Chatbot:  "It looks like there might be some validation issues."

This agent: {
  root_cause: "duplicate_event_id up 50% day-over-day, suggesting upstream replay bug",
  evidence:   ["12 duplicate rows vs 8 yesterday", "event IDs EVT-00019, EVT-00031 appear twice"],
  confidence: "high — confirmed with real quarantine row samples"
}
```

---

## Architecture

The system has four files and two worlds:

**World 1 — The Data Pipeline** (deterministic Python)
```
generate_data.py  →  simulates an upstream system dropping daily CSV files
pipeline.py       →  validates rows, separates clean from bad, writes outputs
```

**World 2 — The AI Agent** (Claude reasoning over evidence)
```
tools.py          →  four functions that read pipeline outputs on demand
agent_loop.py     →  sends question + tools to Claude, executes tool calls, collects answer
```

**The handoff between worlds:**
```
pipeline.py writes files  →  tools.py reads those same files  →  agent reasons over results
```

The pipeline and the agent never talk directly. They communicate through files. This means each part can be built, tested, and replaced independently.

---

## The Full Flow

```
Step 1 — generate_data.py runs
         Creates 500 rows per day with ~10% intentionally bad data
         Writes to data/raw/YYYY-MM-DD.csv

Step 2 — pipeline.py runs
         Reads raw CSV, validates every row against 5 rules
         Writes 4 outputs:
           data/curated/        clean rows only
           data/quarantine/     bad rows labeled with failure reason
           data/summaries/      JSON stats about the run
           data/logs/           timestamped audit trail

Step 3 — You ask a question
         Agent calls tools in whatever order makes sense
         Tools read the pipeline outputs
         Agent produces a structured answer
```

In [ ]:
# Install dependencies
# anthropic  — the Claude API SDK
# boto3      — AWS SDK (used in the cloud version, not needed for this notebook)

!pip install anthropic --quiet
print("Dependencies installed.")

In [ ]:
# Create the local folder structure the pipeline will write to
import os

folders = ["data/raw", "data/curated", "data/quarantine", "data/summaries", "data/logs"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folder structure ready:")
for folder in folders:
    print(f"  {folder}/")

---

## Part 1 — generate_data.py

### What It Does

Simulates an upstream system that drops a CSV file of transaction events every day. In a real system this would be a database export, an S3 file drop, or a Kafka consumer output.

We simulate it locally so the pipeline has something real to process.

### Why We Inject Bad Data

A pipeline that only sees clean data teaches you nothing. We intentionally inject five types of bad rows so the pipeline has something real to catch and the agent has something real to investigate.

| Flaw | What It Simulates |
|---|---|
| missing_user_id | upstream system dropped the user field |
| negative_amount | sign inversion bug in transaction system |
| future_timestamp | clock skew or pre-scheduled event bug |
| invalid_status | upstream using an undocumented status code |
| duplicate_event_id | upstream replay or double-submit bug |

In [ ]:
import csv
import os
import random
from datetime import datetime, timedelta

VALID_STATUSES  = ["success", "failure", "pending"]
ROWS_PER_DAY    = 500
BAD_ROW_FRACTION = 0.10

def make_clean_row(event_num: int, base_date: datetime) -> dict:
    random_seconds = random.randint(0, 86_399)
    timestamp = base_date + timedelta(seconds=random_seconds)
    return {
        "event_id":  f"EVT-{event_num:05d}",
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "user_id":   f"USR-{random.randint(1000, 9999)}",
        "amount":    round(random.uniform(1.00, 999.99), 2),
        "status":    random.choice(VALID_STATUSES),
    }

def inject_flaw(row: dict, flaw_type: str, base_date: datetime) -> dict:
    if flaw_type == "missing_user_id":
        row["user_id"] = ""
    elif flaw_type == "negative_amount":
        row["amount"] = round(random.uniform(-999.99, -0.01), 2)
    elif flaw_type == "future_timestamp":
        future = base_date + timedelta(days=random.randint(1, 30))
        row["timestamp"] = future.strftime("%Y-%m-%d %H:%M:%S")
    elif flaw_type == "invalid_status":
        row["status"] = random.choice(["unknown", "error", "cancelled", ""])
    elif flaw_type == "duplicate_event_id":
        row["event_id"] = f"EVT-{random.randint(1, 50):05d}"
    return row

def generate_raw_file(date_str: str) -> str:
    base_date   = datetime.strptime(date_str, "%Y-%m-%d")
    output_path = os.path.join("data/raw", f"{date_str}.csv")
    total_bad   = int(ROWS_PER_DAY * BAD_ROW_FRACTION)
    bad_indices = set(random.sample(range(ROWS_PER_DAY), total_bad))
    flaw_types  = ["missing_user_id", "negative_amount", "future_timestamp",
                   "invalid_status", "duplicate_event_id"]
    rows = []
    for i in range(ROWS_PER_DAY):
        row = make_clean_row(event_num=i + 1, base_date=base_date)
        if i in bad_indices:
            flaw = flaw_types[len(rows) % len(flaw_types)]
            row  = inject_flaw(row, flaw, base_date)
        rows.append(row)
    fieldnames = ["event_id", "timestamp", "user_id", "amount", "status"]
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return output_path

# Generate 5 days of raw data
today = datetime.today()
dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(4, -1, -1)]

for date_str in dates:
    path = generate_raw_file(date_str)
    print(f"Generated: {path}")

In [ ]:
# Preview the raw data — clean rows and bad rows side by side
import pandas as pd

latest_date = dates[-1]
df = pd.read_csv(f"data/raw/{latest_date}.csv")

print(f"Raw file for {latest_date}: {len(df)} rows")
print()
print("Sample clean rows:")
clean = df[df["user_id"].notna() & (df["user_id"] != "") & (df["amount"] > 0)]
print(clean.head(3).to_string(index=False))
print()
print("Sample bad rows (missing user_id or negative amount):")
bad = df[(df["user_id"] == "") | (df["amount"] < 0)]
print(bad.head(5).to_string(index=False))

---

## Part 2 — pipeline.py

### What It Does

Reads the raw CSV, validates every row against five rules, and routes each row to either the curated file (clean) or the quarantine file (bad). Then writes a JSON summary and a log file.

### The Pipeline Is a Gatekeeper, Not a Cleaner

This is a critical design decision. The pipeline **never fixes bad data**. It only separates and labels it.

If we auto-corrected a missing `user_id` by filling in a default value, downstream analytics would run on invented data without knowing it. That is worse than having no data at all.

Bad rows go to quarantine with a `failure_reason` column so engineers can trace the problem back to the upstream source and fix it at the root.

### The Five Validation Rules

```
Rule 1: user_id cannot be empty         → missing_user_id
Rule 2: amount must be greater than 0   → negative_amount
Rule 3: timestamp cannot be future      → future_timestamp
Rule 4: status must be valid enum       → invalid_status
Rule 5: event_id must be unique         → duplicate_event_id
```

Each row gets the **first rule it breaks**. One clear reason per row.

In [ ]:
import csv
import json
import logging
import os
from collections import defaultdict
from datetime import datetime

VALID_STATUSES = {"success", "failure", "pending"}

def validate_row(row: dict, seen_event_ids: set, run_date: datetime):
    if not row.get("user_id", "").strip():
        return "missing_user_id"
    try:
        if float(row["amount"]) <= 0:
            return "negative_amount"
    except (ValueError, KeyError):
        return "negative_amount"
    try:
        event_time     = datetime.strptime(row["timestamp"], "%Y-%m-%d %H:%M:%S")
        end_of_run_day = run_date.replace(hour=23, minute=59, second=59)
        if event_time > end_of_run_day:
            return "future_timestamp"
    except (ValueError, KeyError):
        return "future_timestamp"
    if row.get("status", "").strip() not in VALID_STATUSES:
        return "invalid_status"
    event_id = row.get("event_id", "").strip()
    if event_id in seen_event_ids:
        return "duplicate_event_id"
    return None

def run_pipeline(date_str: str) -> dict:
    run_start = datetime.now()
    run_date  = datetime.strptime(date_str, "%Y-%m-%d")
    log_lines = []

    def log(msg):
        line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} [INFO] {msg}"
        log_lines.append(line)

    log(f"Pipeline starting for date: {date_str}")

    # Step 1: Read raw file
    raw_path = f"data/raw/{date_str}.csv"
    with open(raw_path, newline="") as f:
        raw_rows = list(csv.DictReader(f))
    log(f"Read {len(raw_rows)} raw rows")

    # Step 2: Validate every row
    curated_rows    = []
    quarantine_rows = []
    seen_event_ids  = set()
    failure_counts  = defaultdict(int)

    for row in raw_rows:
        reason = validate_row(row, seen_event_ids, run_date)
        if reason:
            row["failure_reason"] = reason
            quarantine_rows.append(row)
            failure_counts[reason] += 1
        else:
            curated_rows.append(row)
            seen_event_ids.add(row["event_id"])

    log(f"Validation complete | curated={len(curated_rows)} quarantined={len(quarantine_rows)}")
    for reason, count in sorted(failure_counts.items()):
        log(f"  Failure reason: {reason} = {count} rows")

    # Step 3: Write curated CSV
    os.makedirs("data/curated", exist_ok=True)
    with open(f"data/curated/{date_str}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["event_id","timestamp","user_id","amount","status"])
        writer.writeheader()
        writer.writerows(curated_rows)

    # Step 4: Write quarantine CSV
    os.makedirs("data/quarantine", exist_ok=True)
    with open(f"data/quarantine/{date_str}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["event_id","timestamp","user_id","amount","status","failure_reason"], extrasaction="ignore")
        writer.writeheader()
        writer.writerows(quarantine_rows)

    # Step 5: Write summary JSON
    total_rows      = len(raw_rows)
    quarantine_rate = round(len(quarantine_rows) / total_rows, 4) if total_rows else 0
    summary = {
        "date":             date_str,
        "total_rows":       total_rows,
        "curated_rows":     len(curated_rows),
        "quarantined_rows": len(quarantine_rows),
        "quarantine_rate":  quarantine_rate,
        "failure_reasons":  dict(failure_counts),
        "status":           "completed",
        "run_timestamp":    run_start.strftime("%Y-%m-%d %H:%M:%S"),
    }
    os.makedirs("data/summaries", exist_ok=True)
    with open(f"data/summaries/{date_str}.json", "w") as f:
        json.dump(summary, f, indent=2)

    # Step 6: Write log
    log(f"Pipeline finished | quarantine_rate={quarantine_rate:.1%}")
    os.makedirs("data/logs", exist_ok=True)
    with open(f"data/logs/{date_str}.log", "w") as f:
        f.write("\n".join(log_lines))

    return summary

# Run the pipeline for all 5 days
for date_str in dates:
    summary = run_pipeline(date_str)
    print(f"{date_str} | total={summary['total_rows']} curated={summary['curated_rows']} quarantined={summary['quarantined_rows']} rate={summary['quarantine_rate']:.1%}")

In [ ]:
# Show the summary JSON the pipeline produced
with open(f"data/summaries/{dates[-1]}.json") as f:
    summary = json.load(f)

print(f"Summary for {dates[-1]}:")
print(json.dumps(summary, indent=2))

In [ ]:
# Show sample quarantine rows — labeled with exactly why they failed
df_q = pd.read_csv(f"data/quarantine/{dates[-1]}.csv")

print(f"Quarantine file: {len(df_q)} rows")
print()
print("Failure reason breakdown:")
print(df_q["failure_reason"].value_counts().to_string())
print()
print("Sample quarantined rows:")
print(df_q.head(8).to_string(index=False))

---

## Part 3 — tools.py

### What It Does

Four Python functions that read pipeline outputs and return structured data. These are the **only way the agent is allowed to learn about what happened**.

### Why Tools Exist as a Separate Layer

The agent cannot open files directly. It can only call functions you explicitly give it access to. This boundary is intentional:

- You control exactly what the agent can see
- You control the format of what it receives
- The agent cannot accidentally read sensitive files or raw data

This is the same pattern used in production agent systems — LangChain, OpenAI function calling, Claude tool use, MCP servers. You built it from scratch which means you understand what those frameworks are actually doing.

### The Four Tools

| Tool | Question It Answers |
|---|---|
| `get_run_summary(date)` | How did this run perform overall? |
| `get_log_snippets(date, keyword)` | Did anything unusual happen during the run? |
| `get_quarantine_samples(date, n)` | What do the actual bad rows look like? |
| `compare_runs(date_a, date_b)` | Is this worse than yesterday? |

In [ ]:
import csv
import json
import os

def get_run_summary(date: str) -> dict:
    path = f"data/summaries/{date}.json"
    if not os.path.exists(path):
        raise FileNotFoundError(f"No summary found for {date}")
    with open(path) as f:
        return json.load(f)

def get_log_snippets(date: str, keyword: str = "", max_lines: int = 20) -> list:
    path = f"data/logs/{date}.log"
    if not os.path.exists(path):
        raise FileNotFoundError(f"No log found for {date}")
    with open(path) as f:
        lines = [line.rstrip() for line in f.readlines()]
    if keyword:
        lines = [l for l in lines if keyword.lower() in l.lower()]
    return lines[:max_lines]

def get_quarantine_samples(date: str, n: int = 10, failure_reason: str = "") -> list:
    path = f"data/quarantine/{date}.csv"
    if not os.path.exists(path):
        raise FileNotFoundError(f"No quarantine file found for {date}")
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))
    if failure_reason:
        rows = [r for r in rows if r.get("failure_reason") == failure_reason]
    return rows[:n]

def compare_runs(date_a: str, date_b: str) -> dict:
    summary_a   = get_run_summary(date_a)
    summary_b   = get_run_summary(date_b)
    rate_change = round(summary_b["quarantine_rate"] - summary_a["quarantine_rate"], 4)
    reasons_a   = summary_a.get("failure_reasons", {})
    reasons_b   = summary_b.get("failure_reasons", {})
    all_reasons = set(reasons_a.keys()) | set(reasons_b.keys())
    changes = {}
    for reason in sorted(all_reasons):
        a = reasons_a.get(reason, 0)
        b = reasons_b.get(reason, 0)
        changes[reason] = {date_a: a, date_b: b, "change": b - a}
    return {
        "date_a": date_a, "date_b": date_b,
        "quarantine_rate_change": rate_change,
        "failure_reason_changes": changes,
        "summary_a": summary_a, "summary_b": summary_b,
    }

print("Tools ready.")

In [ ]:
# Demonstrate each tool

print("=== Tool 1: get_run_summary ===")
summary = get_run_summary(dates[-1])
print(json.dumps(summary, indent=2))

print()
print("=== Tool 2: get_log_snippets ===")
lines = get_log_snippets(dates[-1], keyword="Failure")
for line in lines:
    print(line)

print()
print("=== Tool 3: get_quarantine_samples (n=3) ===")
samples = get_quarantine_samples(dates[-1], n=3)
for row in samples:
    print(row)

print()
print("=== Tool 4: compare_runs ===")
diff = compare_runs(dates[-2], dates[-1])
print(f"quarantine_rate_change: {diff['quarantine_rate_change']}")
for reason, data in diff["failure_reason_changes"].items():
    print(f"  {reason}: {data}")

---

## Part 4 — agent_loop.py

### What It Does

Sends your question to Claude along with the tool definitions and a system prompt. Executes whatever tool calls Claude requests. Loops until Claude has enough evidence to produce the final structured answer.

### The Three Parts of agent_loop.py

**1. The System Prompt**
Instructions Claude reads before every question. Tells it to behave as a pipeline ops assistant, call tools before drawing conclusions, and return a specific JSON schema.

**2. The Tool Schemas**
Descriptions of each tool — what it does, when to use it, what arguments it takes. Claude reads these to decide which tools are relevant to the question.

**3. The Agent Loop**
```
Send question + tools to Claude
        │
Claude responds with tool_use  →  execute tool  →  send result back  →  loop
        │
Claude responds with end_turn  →  parse JSON answer  →  return
```

### What You Program vs What Claude Does

```
You program                      Claude does
────────────────────────────     ──────────────────────────────────
the tool functions               decides which tools to call
the tool descriptions            reads evidence from tool results
the system prompt rules          identifies patterns across results
the output schema                fills in the schema with conclusions
the dispatcher (routes calls)    reasons about root cause
```

### To Run the Agent

You need an Anthropic API key. Add it as a Kaggle secret named `ANTHROPIC_API_KEY`:
1. Click the lock icon in the sidebar
2. Add a new secret: name = `ANTHROPIC_API_KEY`, value = your key
3. Enable it for this notebook

In [ ]:
import json
import os
import anthropic
from kaggle_secrets import UserSecretsClient

# Load API key from Kaggle secrets
# If running locally, set ANTHROPIC_API_KEY as an environment variable instead
try:
    secrets = UserSecretsClient()
    os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
    print("API key loaded from Kaggle secrets.")
except Exception:
    print("Running locally — using ANTHROPIC_API_KEY from environment.")

In [ ]:
SYSTEM_PROMPT = """
You are an expert data pipeline operations assistant.

Your job is to investigate pipeline run issues by reading evidence from tools.
You must never state a root cause or make a recommendation without first calling
tools to gather supporting evidence.

Investigation rules:
- Always call get_run_summary first to understand the overall picture
- If the quarantine rate is elevated, call get_quarantine_samples
- If the question involves change over time, call compare_runs
- If you need to understand the sequence of events, call get_log_snippets

When you have gathered enough evidence, respond with ONLY a JSON object:
{
  "summary": "one paragraph plain English explanation",
  "root_cause": "the specific technical finding",
  "evidence": ["fact 1 from tool output", "fact 2 from tool output"],
  "confidence": "high | medium | low — one sentence why",
  "recommended_fixes": ["1. first action", "2. second action"],
  "next_checks": ["what to check if fixes don't work"]
}
"""

TOOL_SCHEMAS = [
    {
        "name": "get_run_summary",
        "description": "Returns the full summary for a pipeline run on a given date. Use this first for any question about a specific run.",
        "input_schema": {"type": "object", "properties": {"date": {"type": "string", "description": "Run date in YYYY-MM-DD format"}}, "required": ["date"]},
    },
    {
        "name": "get_log_snippets",
        "description": "Returns log lines that match a keyword. Use to find errors or specific events. Good keywords: ERROR, quarantined, negative_amount.",
        "input_schema": {"type": "object", "properties": {"date": {"type": "string"}, "keyword": {"type": "string"}, "max_lines": {"type": "integer"}}, "required": ["date"]},
    },
    {
        "name": "get_quarantine_samples",
        "description": "Returns sample quarantined rows. Optionally filter by failure_reason. Valid reasons: missing_user_id, negative_amount, future_timestamp, invalid_status, duplicate_event_id.",
        "input_schema": {"type": "object", "properties": {"date": {"type": "string"}, "n": {"type": "integer"}, "failure_reason": {"type": "string"}}, "required": ["date"]},
    },
    {
        "name": "compare_runs",
        "description": "Compares two pipeline runs. Use when the question involves change over time. date_a is baseline, date_b is current.",
        "input_schema": {"type": "object", "properties": {"date_a": {"type": "string"}, "date_b": {"type": "string"}}, "required": ["date_a", "date_b"]},
    },
]

def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    print(f"  [tool call] {tool_name}({tool_input})")
    if tool_name == "get_run_summary":
        result = get_run_summary(**tool_input)
    elif tool_name == "get_log_snippets":
        result = get_log_snippets(**tool_input)
    elif tool_name == "get_quarantine_samples":
        result = get_quarantine_samples(**tool_input)
    elif tool_name == "compare_runs":
        result = compare_runs(**tool_input)
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
    return json.dumps(result, indent=2)

def run_agent(question: str) -> dict:
    client   = anthropic.Anthropic()
    messages = [{"role": "user", "content": question}]

    print(f"Question: {question}")
    print("-" * 60)

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOL_SCHEMAS,
            messages=messages,
        )

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result_str = dispatch_tool(block.name, block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result_str})
            messages.append({"role": "user", "content": tool_results})

        elif response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
                    break
            try:
                start  = final_text.find("{")
                end    = final_text.rfind("}") + 1
                return json.loads(final_text[start:end])
            except json.JSONDecodeError:
                return {"summary": final_text, "root_cause": "Could not parse response",
                        "evidence": [], "confidence": "low", "recommended_fixes": [], "next_checks": []}
        else:
            return {"summary": "Unexpected stop", "root_cause": "Unknown",
                    "evidence": [], "confidence": "low", "recommended_fixes": [], "next_checks": []}

print("Agent loop ready.")

In [ ]:
# Run the agent — ask it a real operational question
question = f"Why did the {dates[-1]} pipeline run have a high quarantine rate?"
answer   = run_agent(question)

print()
print("=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
print(json.dumps(answer, indent=2))

In [ ]:
# Ask a trend question — uses compare_runs
question2 = f"What changed between {dates[-2]} and {dates[-1]}? Is it getting better or worse?"
answer2   = run_agent(question2)

print()
print("=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
print(json.dumps(answer2, indent=2))

---

## Key Takeaways

### 1. Separate your concerns
The pipeline generates facts. The tools expose facts. The agent interprets facts. Three separate layers, three separate files. Each can be changed without touching the others.

### 2. Label bad data, never fix it silently
Auto-correcting missing values creates invisible data quality debt. Label bad rows with exactly why they failed and quarantine them so the upstream source can be fixed properly.

### 3. Tools make agents trustworthy
An agent that answers from general knowledge can hallucinate. An agent that must call tools before forming conclusions is grounded in evidence you can verify yourself.

### 4. The dispatcher pattern is everywhere
Claude cannot run Python. It requests tool calls by name. Your code executes them and returns results. This is exactly what MCP servers, LangChain tool use, and OpenAI function calling do — just with more infrastructure around the same core pattern.

### 5. Structured output beats prose
A JSON response with root_cause, evidence, confidence, and recommended_fixes is more useful than a paragraph. Different consumers — engineers, managers, automated systems — can pull the field they need.

---

## Cloud Version

This notebook runs everything locally. The production version of this project replaces the local file I/O with AWS S3:

```python
# Local version
with open("data/summaries/2026-04-12.json") as f:
    return json.load(f)

# AWS version — same result, different source
response = s3.get_object(Bucket="my-bucket", Key="summaries/2026-04-12.json")
return json.loads(response["Body"].read().decode("utf-8"))
```

The validation logic, tool functions, and agent loop are identical in both versions. The only thing that changes is where files are read from and written to.

Full source code: [github.com/nalkaslassy/ops-insight-assistant](https://github.com/nalkaslassy/ops-insight-assistant)

---

## Tech Stack

- **Python 3.11+**
- **Claude (claude-sonnet-4-6)** via Anthropic API
- **AWS S3** via boto3 (cloud version)
- **Standard library** — csv, json, logging, io, collections